# SMAP Dataset Visualization

NASA **SMAP** (Soil Moisture Active Passive) satellite telemetry for anomaly detection.

## Data layout

- **53 channels** (`A-1`, `A-2`, … `P-1`, …) are **concatenated** in sorted `chan_id` order (`P-2` excluded).
- **train** (`data/train/*.npy`): per-channel normal segments only (no labels in the official pipeline).
- **test** (`data/test/*.npy`): per-channel test streams.
- **Anomaly labels**: interval `[start, end]` per channel in `data/labeled_anomalies.csv`.
- **Preprocessed** (`processed/SMAP_*.pkl`): all channels concatenated into one array.

This notebook plots a **single channel (P-1)** and a **slice of the concatenated test** set to show how anomalies are mixed in.

In [ ]:
import ast
import csv
import os
import pickle

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

DATA_DIR = 'data'
PROCESSED_DIR = 'processed'
CHANNEL_ID = 'P-1'  # channel to visualize (see labeled_anomalies.csv)

In [ ]:
def load_smap_channels():
    with open(os.path.join(DATA_DIR, 'labeled_anomalies.csv')) as f:
        rows = list(csv.reader(f))[1:]
    channels = sorted(
        [r for r in rows if r[1] == 'SMAP' and r[0] != 'P-2'],
        key=lambda r: r[0],
    )
    return channels


def load_channel(chan_id):
    train = np.load(os.path.join(DATA_DIR, 'train', f'{chan_id}.npy'))
    test = np.load(os.path.join(DATA_DIR, 'test', f'{chan_id}.npy'))
    return train, test


def channel_test_label(row):
    length = int(row[-1])
    label = np.zeros(length, dtype=bool)
    for start, end in ast.literal_eval(row[2]):
        label[start:end + 1] = True
    return label


def channel_offset_in_concat(channels, chan_id, split='test'):
    """Start index of a channel in the concatenated train/test array."""
    offset = 0
    for row in channels:
        if row[0] == chan_id:
            return offset
        if split == 'test':
            offset += int(row[-1])
        else:
            train = np.load(os.path.join(DATA_DIR, 'train', f"{row[0]}.npy"))
            offset += len(train)
    raise KeyError(chan_id)


channels = load_smap_channels()
print(f'SMAP channels: {len(channels)}')
print('First 5:', [c[0] for c in channels[:5]])

In [ ]:
with open(os.path.join(PROCESSED_DIR, 'SMAP_train.pkl'), 'rb') as f:
    train_all = pickle.load(f)
with open(os.path.join(PROCESSED_DIR, 'SMAP_test.pkl'), 'rb') as f:
    test_all = pickle.load(f)
with open(os.path.join(PROCESSED_DIR, 'SMAP_test_label.pkl'), 'rb') as f:
    label_all = pickle.load(f)

print('concatenated train:', train_all.shape)
print('concatenated test :', test_all.shape)
print('test labels       :', label_all.shape)
print(f'test anomaly ratio: {label_all.mean() * 100:.2f}%')
print(f'num features      : {train_all.shape[1]}')

## 1. Single channel (P-1): train tail + test head

Red shading = **anomaly intervals** from `labeled_anomalies.csv` (indices in the test stream).

In [ ]:
row = next(r for r in channels if r[0] == CHANNEL_ID)
train_ch, test_ch = load_channel(CHANNEL_ID)
label_ch = channel_test_label(row)
anomaly_spans = ast.literal_eval(row[2])

print(CHANNEL_ID, 'train', train_ch.shape, 'test', test_ch.shape)
print('anomaly spans (test index):', anomaly_spans)
print('anomaly ratio:', f'{label_ch.mean() * 100:.2f}%')

TRAIN_TAIL = 1500
TEST_HEAD = 5000
dim = 0  # feature index to plot

train_seg = train_ch[-TRAIN_TAIL:, dim]
test_seg = test_ch[:TEST_HEAD, dim]

t_train = np.arange(-len(train_seg), 0)
t_test = np.arange(0, len(test_seg))

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(t_train, train_seg, color='steelblue', lw=0.9, label='train (assumed normal)')
ax.plot(t_test, test_seg, color='black', lw=0.9, label='test')
ax.axvspan(0, TEST_HEAD, color='lightgray', alpha=0.15)
for start, end in anomaly_spans:
    if start >= TEST_HEAD:
        break
    ax.axvspan(start, min(end, TEST_HEAD - 1), color='red', alpha=0.25)
ax.axvline(0, color='gray', ls='--', lw=1)
ax.set_title(f'SMAP {CHANNEL_ID} — feature {dim} (train tail + test head)')
ax.set_xlabel('time index (0 = start of test)')
ax.set_ylabel('value (pre-scaled)')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
# Full P-1 test: label timeline + one feature
fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True,
                        gridspec_kw={'height_ratios': [1, 2]})
dim = 0
axes[0].fill_between(np.arange(len(label_ch)), 0, label_ch.astype(int),
                     step='mid', color='crimson', alpha=0.6, label='anomaly')
axes[0].set_ylabel('label')
axes[0].set_ylim(-0.1, 1.3)
axes[0].set_title(f'{CHANNEL_ID} test — anomaly label timeline')
axes[0].legend(loc='upper right')

axes[1].plot(test_ch[:, dim], color='black', lw=0.7)
for start, end in anomaly_spans:
    axes[1].axvspan(start, end, color='red', alpha=0.2)
axes[1].set_ylabel(f'feature {dim}')
axes[1].set_xlabel('test time index')
plt.tight_layout()
plt.show()

## 2. P-1 inside the concatenated test array

`processed/SMAP_test.pkl` used by OmniAnomaly concatenates all 53 channel test streams in order.

In [ ]:
offset = channel_offset_in_concat(channels, CHANNEL_ID, 'test')
length = int(next(r for r in channels if r[0] == CHANNEL_ID)[-1])
print(f'{CHANNEL_ID} in concatenated test: [{offset}, {offset + length})')

WIN = 2000
start = offset + 2000
end = start + WIN
seg = test_all[start:end]
seg_label = label_all[start:end]
dim = 0

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(seg[:, dim], color='black', lw=0.8)
anomaly_idx = np.where(seg_label)[0]
if len(anomaly_idx):
    ax.scatter(anomaly_idx, seg[anomaly_idx, dim], s=8, c='red', alpha=0.5, label='anomaly points')
    in_anom = False
    s0 = 0
    for i, v in enumerate(seg_label):
        if v and not in_anom:
            s0 = i; in_anom = True
        elif not v and in_anom:
            ax.axvspan(s0, i - 1, color='red', alpha=0.15); in_anom = False
    if in_anom:
        ax.axvspan(s0, len(seg_label) - 1, color='red', alpha=0.15)
ax.set_title(f'Concatenated test [{start}:{end}) — inside {CHANNEL_ID}, feature {dim}')
ax.set_xlabel('index within window')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Train vs test anomaly ratio

Train has no labels (all assumed normal). About **13%** of test points are labeled anomalous.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

bounds = [0]
ratios = []
names = []
for row in channels:
    n = int(row[-1])
    bounds.append(bounds[-1] + n)
    sl = slice(bounds[-2], bounds[-1])
    ratios.append(label_all[sl].mean())
    names.append(row[0])

axes[0].bar(range(len(ratios)), ratios, color='indianred', alpha=0.7)
axes[0].set_xlabel('channel index (sorted)')
axes[0].set_ylabel('anomaly ratio')
axes[0].set_title('SMAP test — anomaly ratio per channel')

train_slice = train_all[:3000]
test_slice = test_all[offset:offset + 3000]
test_lbl_slice = label_all[offset:offset + 3000]

vmax = max(train_slice.max(), test_slice.max())
vmin = min(train_slice.min(), test_slice.min())

im0 = axes[1].imshow(train_slice.T, aspect='auto', cmap='viridis', vmin=vmin, vmax=vmax)
axes[1].set_title('Train — first 3000 steps (25 features)')
axes[1].set_xlabel('time')
axes[1].set_ylabel('feature')
plt.colorbar(im0, ax=axes[1], fraction=0.046)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(12, 4))
im = ax.imshow(test_slice.T, aspect='auto', cmap='viridis', vmin=vmin, vmax=vmax)
for i, v in enumerate(test_lbl_slice):
    if v:
        ax.axvline(i, color='red', alpha=0.08, lw=2)
ax.set_title(f'{CHANNEL_ID} test — first 3000 steps (red lines = anomaly)')
ax.set_xlabel('time')
ax.set_ylabel('feature')
plt.colorbar(im, fraction=0.046)
plt.tight_layout()
plt.show()